<a href="https://colab.research.google.com/github/lydiacyhung/114-2-ProgramingLanguage/blob/main/HW4_%E6%96%87%E5%AD%97%E8%B3%87%E6%96%99%E5%B0%8F%E5%88%86%E6%9E%90_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss.cpu

In [ ]:
import os, time, uuid, re, json, datetime
from datetime import datetime as dt, timedelta
from dateutil.tz import gettz
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup
import numpy as np # Added numpy import for FAISS operations

import google.generativeai as genai

# Google Auth & Sheets
from google.colab import auth
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.auth import default

# RAG specific imports
import faiss

In [ ]:
!pip -q install faiss-cpu

In [ ]:
import faiss

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [ ]:
from google.colab import userdata

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('Gemini')

# 使用獲取的金鑰配置 genai
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-3.1-flash-lite')

In [ ]:
embedding_model_name = "models/embedding-001"
def get_embedding(text):
    if not text:
        return []
    try:
        # Using text-embedding-004 which is the recommended model for general tasks
        result = genai.embed_content(
            model=embedding_model_name,
            content=str(text),
            task_type="retrieval_document"
        )
        return result["embedding"]
    except Exception as e:
        print(f"Embedding error: {e}")
        return []

In [ ]:
# Global variables for FAISS index and document mapping
faiss_index = None
rag_index_to_doc_map = []

In [ ]:
# Global variables for FAISS index and document mapping
faiss_index = None
rag_index_to_doc_map = []

In [ ]:
PTT_HEADER = [
    "post_id","title","url","date","author","nrec","created_at",
    "fetched_at","content"
]
TERMS_HEADER = ["term","freq","df_count","tfidf_mean","examples"]

In [ ]:
PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/movie/index.html"

In [ ]:
def ensure_spreadsheet(name):
    try:
        sh = gc.open(name)  # returns gspread.models.Spreadsheet
    except gspread.SpreadsheetNotFound:
        sh = gc.create(name)
    return sh

sh = ensure_spreadsheet(WORKSHEET_NAME)

In [ ]:
def ensure_worksheet(sh, title, header):
    try:
        ws = sh.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=title, rows="1000", cols=str(len(header)+5))
        ws.update([header])
    # 若沒有表頭就補上
    data = ws.get_all_values()
    if not data or (data and data[0] != header):
        ws.clear()
        ws.update([header])
    return ws

In [ ]:
ws_ptt_posts = ensure_worksheet(sh, "ptt_movie_posts", PTT_HEADER)
ws_ptt_terms = ensure_worksheet(sh, "ptt_movie_terms", TERMS_HEADER)

In [ ]:
try:
    # 1. 開啟試算表
    sh = gc.open_by_url(SHEET_URL)
    print(f"成功連接試算表: {sh.title}")

    # 2. 檢查並建立工作表
    ws_list = [w.title for w in sh.worksheets()]
    print(f"目前現有的分頁: {ws_list}")

    target_ws_name = "HW4"
    if target_ws_name not in ws_list:
        print(f"正在建立缺失的分頁: {target_ws_name}...")
        ws_ptt_posts = sh.add_worksheet(title=target_ws_name, rows="1000", cols=str(len(PTT_HEADER)+2))
        ws_ptt_posts.update([PTT_HEADER])
    else:
        ws_ptt_posts = sh.worksheet(target_ws_name)

    # 3. 寫入現有資料
    if not ptt_posts_df.empty:
        print(f"正在將本地 {len(ptt_posts_df)} 筆資料寫入 {target_ws_name}...")
        write_df(ws_ptt_posts, ptt_posts_df, PTT_HEADER)
        print("✅ 寫入完成！請檢查 Google Sheet。")
    else:
        print("ℹ️ 本地目前無爬蟲資料，請執行爬蟲儲存格後再試。")

except Exception as e:
    print(f"❌ 處理過程發生錯誤: {e}")

### 自動化流程：抓取 PTT 並直接寫入 Google Sheet (HW4)

In [ ]:
def crawl_ptt_movie(index_pages=3, min_push=0, keyword="", target_ws=None):
    """從最新 index.html 往前翻 index_pages 頁，抓滿足條件的文章"""
    global ptt_posts_df
    url = PTT_MOVIE_INDEX
    all_rows = []
    # 確保 ptt_posts_df 已初始化
    if 'ptt_posts_df' not in globals() or ptt_posts_df is None:
        import pandas as pd
        ptt_posts_df = pd.DataFrame(columns=PTT_HEADER)

    seen_urls = set(ptt_posts_df["url"].tolist()) if not ptt_posts_df.empty else set()

    for _ in range(int(index_pages)):
        try:
            soup = _get_soup(url)
            posts = _extract_post_list(soup)
        except Exception as e:
            print(f"無法讀取索引頁 {url}: {e}")
            break

        for p in posts:
            if p["nrec"] < int(min_push): continue
            if keyword and (keyword not in p["title"]): continue
            if p["url"] in seen_urls: continue

            try:
                art_soup = _get_soup(p["url"])
                content, meta_title = _clean_ptt_content(art_soup)
                time.sleep(0.1)
            except:
                content, meta_title = "", ""

            final_title = p["title"] if p["title"] else (meta_title or "（無標題）")

            all_rows.append({
                "post_id": str(uuid.uuid4())[:8], "title": final_title[:200], "url": p["url"],
                "date": p["date"], "author": p["author"], "nrec": str(p["nrec"]),
                "created_at": dt.now().isoformat(), "fetched_at": dt.now().isoformat(), "content": content
            })

        prev = _get_prev_index_url(soup)
        if not prev: break
        url = prev

    if all_rows:
        new_df = pd.DataFrame(all_rows, columns=PTT_HEADER)
        ptt_posts_df = pd.concat([ptt_posts_df, new_df], ignore_index=True)
        if target_ws:
            write_df(target_ws, ptt_posts_df, PTT_HEADER)
        return f"✅ 取得 {len(all_rows)} 篇文章", ptt_posts_df
    else:
        return "ℹ️ 沒有新文章符合條件", ptt_posts_df

In [ ]:
TASKS_HEADER = [
    "id","task","status","priority","est_min","start_time","end_time",
    "actual_min","pomodoros","due_date","labels","notes",
    "created_at","updated_at","completed_at","planned_for"
]
LOGS_HEADER = [
    "log_id","task_id","phase","start_ts","end_ts","minutes","cycles","note"
]
CLIPS_HEADER = ["clip_id","url","selector","text","href","created_at","added_to_task"]

ws_tasks = ensure_worksheet(sh, "tasks", TASKS_HEADER)
ws_logs  = ensure_worksheet(sh, "pomodoro_logs", LOGS_HEADER)
ws_clips = ensure_worksheet(sh, "web_clips", CLIPS_HEADER)

def tznow():
    return dt.now(gettz(TIMEZONE))

def read_df(ws, header):
    df = get_as_dataframe(ws, evaluate_formulas=True, header=0)
    if df is None or df.empty:
        return pd.DataFrame(columns=header)
    df = df.fillna("")
    # 保證欄位齊全
    for c in header:
        if c not in df.columns:
            df[c] = ""
    # 型別微調
    if "est_min" in df.columns:
        df["est_min"] = pd.to_numeric(df["est_min"], errors="coerce").fillna(0).astype(int)
    if "actual_min" in df.columns:
        df["actual_min"] = pd.to_numeric(df["actual_min"], errors="coerce").fillna(0).astype(int)
    if "pomodoros" in df.columns:
        df["pomodoros"] = pd.to_numeric(df["pomodoros"], errors="coerce").fillna(0).astype(int)
    return df[header]

def write_df(ws, df, header):
    if df.empty:
        ws.clear()
        ws.update([header])
        return
    # 轉字串避免 gspread 型別問題
    df_out = df.copy()
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)
    ws.clear()
    ws.update([header] + df_out.values.tolist())

def refresh_all():
    return (
        read_df(ws_tasks, TASKS_HEADER).copy(),
        read_df(ws_logs, LOGS_HEADER).copy(),
        read_df(ws_clips, CLIPS_HEADER).copy()
    )

tasks_df, logs_df, clips_df = refresh_all()

def add_task(task, priority, est_min, due_date, labels, notes, planned_for):
    global tasks_df
    _now = tznow().isoformat()
    new = pd.DataFrame([{
        "id": str(uuid.uuid4())[:8],
        "task": task.strip(),
        "status": "todo",
        "priority": priority or "M",
        "est_min": int(est_min) if est_min else 25,
        "start_time": "",
        "end_time": "",
        "actual_min": 0,
        "pomodoros": 0,
        "due_date": due_date or "",
        "labels": labels or "",
        "notes": notes or "",
        "created_at": _now,
        "updated_at": _now,
        "completed_at": "",
        "planned_for": planned_for or ""  # 可填 today / tomorrow / 空白
    }])
    tasks_df = pd.concat([tasks_df, new], ignore_index=True)
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    return "✅ 已新增任務", tasks_df

def update_task_status(task_id, new_status):
    global tasks_df
    idx = tasks_df.index[tasks_df["id"] == task_id]
    if len(idx)==0:
        return "⚠️ 找不到任務", tasks_df
    i = idx[0]
    tasks_df.loc[i, "status"] = new_status
    tasks_df.loc[i, "updated_at"] = tznow().isoformat()
    if new_status == "done" and not tasks_df.loc[i, "completed_at"]:
        tasks_df.loc[i, "completed_at"] = tznow().isoformat()
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    return "✅ 狀態已更新", tasks_df

def mark_done(task_id):
    return update_task_status(task_id, "done")

def recalc_task_actuals(task_id):
    """根據 logs_df 回寫 actual_min 與 pomodoros"""
    global tasks_df, logs_df
    work_logs = logs_df[(logs_df["task_id"]==task_id) & (logs_df["phase"]=="work")]
    total_min = work_logs["minutes"].astype(float).sum() if not work_logs.empty else 0
    pomos = int(round(total_min / 25.0))
    idx = tasks_df.index[tasks_df["id"]==task_id]
    if len(idx)==0: return
    i = idx[0]
    tasks_df.loc[i,"actual_min"] = int(total_min)
    tasks_df.loc[i,"pomodoros"] = pomos
    tasks_df.loc[i,"updated_at"] = tznow().isoformat()

def list_task_choices():
    global tasks_df
    if tasks_df.empty:
        return []
    # 顯示： [status] (P:priority) task  — id
    def row_label(r):
        return f"[{r['status']}] (P:{r['priority']}) {r['task']} — {r['id']}"
    return [(row_label(r), r["id"]) for _, r in tasks_df.iterrows()]

# 我們採「按鈕開始 / 結束」模式（避免後端阻塞），每次按「開始」會先記住 start_ts，
# 按「結束」時計算分鐘並寫入 logs，再回填任務 actual_min / pomodoros。

_active_sessions = {}  # { task_id: {"phase": "work"/"break", "start_ts": iso, "cycles": int} }

def start_phase(task_id, phase, cycles):
    if not task_id: return "⚠️ 請先選擇任務"
    _active_sessions[task_id] = {
        "phase": phase,
        "start_ts": tznow().isoformat(),
        "cycles": int(cycles) if cycles else 1
    }
    return f"▶️ 已開始：{phase}（task: {task_id}）"

def end_phase(task_id, note):
    global logs_df, tasks_df
    if task_id not in _active_sessions:
        return "⚠️ 尚未開始任何階段"
    sess = _active_sessions.pop(task_id)
    start = pd.to_datetime(sess["start_ts"])
    end = tznow()
    minutes = round((end - start).total_seconds() / 60.0, 2)
    log = pd.DataFrame([{
        "log_id": str(uuid.uuid4())[:8],
        "task_id": task_id,
        "phase": sess["phase"],
        "start_ts": start.isoformat(),
        "end_ts": end.isoformat(),
        "minutes": minutes,
        "cycles": int(sess["cycles"]),
        "note": note or ""
    }])
    logs_df = pd.concat([logs_df, log], ignore_index=True)
    write_df(ws_logs, logs_df, LOGS_HEADER)

    # 回填任務
    if sess["phase"] == "work":
        recalc_task_actuals(task_id)
        write_df(ws_tasks, tasks_df, TASKS_HEADER)

    return f"⏹️ 已結束：{sess['phase']}，紀錄 {minutes} 分鐘"

# AI 計畫（Gemini；無金鑰則規則式）
def generate_today_plan():
    global tasks_df
    # 以「due_date 是今天」或「planned_for = today」且不是 done 的任務為計畫清單
    today = tznow().date().isoformat()
    cand = tasks_df[
        ((tasks_df["due_date"]==today) | (tasks_df["planned_for"].str.lower()=="today")) &
        (tasks_df["status"]!="done")
    ].copy()
    if cand.empty:
        return "📭 今天沒有標記的任務。請在 Tasks 分頁把任務的 due_date 設為今天或 planned_for 設為 today。"

    # 先依 priority（H>M>L）+ est_min 排序
    pr_order = {"H":0, "M":1, "L":2}
    cand["p_ord"] = cand["priority"].map(pr_order).fillna(3)
    cand = cand.sort_values(["p_ord","est_min"], ascending=[True, True])

    # 嘗試 Gemini
    api_key = os.environ.get("GEMINI_API_KEY","").strip()
    if api_key:
        genai.configure(api_key=api_key)
        sys_prompt = (
            "你是一位任務規劃助理。請把輸入的任務（含估時與優先級）排成三段：morning、afternoon、evening，"
            "並給出每段的重點、順序、每項的時間預估與備註。總時數請大致符合任務估時總和。"
            "回傳以 Markdown 條列，格式：\n"
            "### Morning\n- [任務ID] 任務名稱（預估 xx 分）— 備註\n..."
            "### Afternoon\n...\n### Evening\n...\n"
        )
        items = []
        for _, r in cand.iterrows():
            items.append({
                "id": r["id"], "task": r["task"], "est_min": int(r["est_min"]),
                "priority": r["priority"]
            })
        user_content = json.dumps({"today": today, "tasks": items}, ensure_ascii=False)
        try:
            model = genai.GenerativeModel("gemini-1.5-flash")
            resp = model.generate_content(sys_prompt + "\n\n" + user_content)
            plan_md = resp.text
        except Exception as e:
            plan_md = f"⚠️ Gemini 失敗：{e}\n\n改用規則式規劃。"
    else:
        plan_md = "🔧 未設定 GEMINI_API_KEY，使用規則式規劃。\n\n"

    # 規則式：把高優先任務平均切到上午/下午/晚上
    buckets = {"morning": [], "afternoon": [], "evening": []}
    total = len(cand)
    for i, (_, r) in enumerate(cand.iterrows()):
        if i % 3 == 0:
            buckets["morning"].append(r)
        elif i % 3 == 1:
            buckets["afternoon"].append(r)
        else:
            buckets["evening"].append(r)

    def sec_md(name, rows):
        if not rows: return f"### {name.title()}\n（無）\n"
        lines = [f"### {name.title()}"]
        for r in rows:
            lines.append(f"- [{r['id']}] {r['task']}（預估 {int(r['est_min'])} 分，P:{r['priority']}）")
        return "\n".join(lines) + "\n"

    rule_md = sec_md("morning", buckets["morning"]) + "\n" + \
              sec_md("afternoon", buckets["afternoon"]) + "\n" + \
              sec_md("evening", buckets["evening"])

    return (plan_md + "\n---\n" + rule_md).strip()

# 今日完成率
def today_summary():
    global tasks_df
    today = tznow().date().isoformat()
    planned = tasks_df[
        ((tasks_df["due_date"]==today) | (tasks_df["planned_for"].str.lower()=="today"))
    ]
    done = planned[planned["status"]=="done"]
    total = len(planned)
    done_n = len(done)
    rate = (done_n/total*100) if total>0 else 0
    return f"📅 今日計畫任務：{total}；✅ 完成：{done_n}；📈 完成率：{rate:.1f}%"

# =========================
# 爬蟲：擷取文字或連結並可加入任務
# =========================
def crawl(url, selector, mode, limit):
    try:
        resp = requests.get(url, timeout=15, headers={"User-Agent":"Mozilla/5.0"})
        resp.raise_for_status()
    except Exception as e:
        return pd.DataFrame(columns=CLIPS_HEADER), f"⚠️ 請求失敗：{e}"

    soup = BeautifulSoup(resp.text, "html.parser")
    nodes = soup.select(selector)
    rows = []
    for i, n in enumerate(nodes[:int(limit) if limit else 20]):
        text = n.get_text(strip=True) if mode in ("text","both") else ""
        href = n.get("href") if mode in ("href","both") else ""
        # 相對連結處理
        if href and href.startswith("/"):
            from urllib.parse import urljoin
            href = urljoin(url, href)
        rows.append({
            "clip_id": str(uuid.uuid4())[:8],
            "url": url,
            "selector": selector,
            "text": text,
            "href": href,
            "created_at": tznow().isoformat(),
            "added_to_task": ""
        })
    df = pd.DataFrame(rows, columns=CLIPS_HEADER)
    return df, f"✅ 擷取 {len(df)} 筆"

def add_clips_as_tasks(clip_ids, default_priority, est_min):
    global clips_df, tasks_df
    if not clip_ids:
        return "⚠️ 請先勾選要加入的爬蟲項目", clips_df, tasks_df
    sel = clips_df[clips_df["clip_id"].isin(clip_ids)]
    _now = tznow().isoformat()
    new_tasks = []
    for _, r in sel.iterrows():
        title = r["text"] or r["href"] or "（未命名）"
        note = f"來源：{r['url']}\n選擇器：{r['selector']}\n連結：{r['href']}"
        new_tasks.append({
            "id": str(uuid.uuid4())[:8],
            "task": title[:120],
            "status": "todo",
            "priority": default_priority or "M",
            "est_min": int(est_min) if est_min else 25,
            "start_time": "",
            "end_time": "",
            "actual_min": 0,
            "pomodoros": 0,
            "due_date": "",
            "labels": "from:crawler",
            "notes": note,
            "created_at": _now,
            "updated_at": _now,
            "completed_at": "",
            "planned_for": ""
        })
    if new_tasks:
        tasks_df = pd.concat([tasks_df, pd.DataFrame(new_tasks)], ignore_index=True)
        # 標記已加入
        clips_df.loc[clips_df["clip_id"].isin(clip_ids), "added_to_task"] = "yes"
        write_df(ws_tasks, tasks_df, TASKS_HEADER)
        write_df(ws_clips, clips_df, CLIPS_HEADER)
        return f"✅ 已加入 {len(new_tasks)} 項為任務", clips_df, tasks_df
    return "⚠️ 無可加入項目", clips_df, tasks_df


def read_ptt_posts_df():
    return read_df(ws_ptt_posts, PTT_HEADER).copy()

def read_terms_df():
    return read_df(ws_ptt_terms, TERMS_HEADER).copy()

ptt_posts_df = read_ptt_posts_df()
terms_df = read_terms_df()


In [ ]:
def build_faiss_index_from_ptt_data():
    global faiss_index, rag_index_to_doc_map, ptt_posts_df

    if ptt_posts_df.empty:
        return "⚠️ PTT 文章資料為空，請先在 'PTT 爬蟲' 分頁爬取文章。", None

    print("正在生成文章 Embedding 並建立 FAISS 索引...")
    embeddings = []
    documents = []

    for idx, row in ptt_posts_df.iterrows():
        # Combine title and content for better context
        combined_text = f"標題: {row['title']}\n內容: {row['content']}"
        embedding = get_embedding(combined_text)
        if embedding:
            embeddings.append(embedding)
            documents.append({
                "post_id": row['post_id'],
                "title": row['title'],
                "url": row['url'],
                "content": row['content']
            })

    if not embeddings:
        return "⚠️ 無法生成任何文章 Embedding，請檢查模型和輸入內容。", None

    # Convert list of embeddings to a NumPy array
    embeddings_array = np.array(embeddings).astype('float32')

    # Get embedding dimension
    d = embeddings_array.shape[1]

    # Create a FAISS index (e.g., IndexFlatL2 for L2 distance, good for cosine similarity if normalized)
    faiss_index = faiss.IndexFlatL2(d)

    # Add embeddings to the index
    faiss_index.add(embeddings_array)

    rag_index_to_doc_map = documents
    print(f"已成功建立 FAISS 索引，包含 {len(documents)} 篇文章。")
    return f"✅ 已從 {len(documents)} 篇文章建立 FAISS 索引。", faiss_index

In [ ]:
def retrieve_documents_from_faiss(query_text, top_k=3):
    global faiss_index, rag_index_to_doc_map

    if faiss_index is None or faiss_index.ntotal == 0:
        return [], "⚠️ FAISS 索引尚未建立或為空，請先建立索引。"
    if not query_text:
        return [], "⚠️ 查詢內容不可為空。"

    try:
        query_embedding = np.array(get_embedding(query_text)).astype('float32').reshape(1, -1)
    except Exception as e:
        return [], f"⚠️ 查詢 Embedding 失敗: {e}"

    # Perform search
    distances, indices = faiss_index.search(query_embedding, top_k)

    retrieved_docs = []
    for i, doc_idx in enumerate(indices[0]):
        if doc_idx != -1: # -1 indicates no result found
            doc = rag_index_to_doc_map[doc_idx]
            doc['distance'] = distances[0][i] # Add distance for debugging/info
            retrieved_docs.append(doc)

    if not retrieved_docs:
        return [], "ℹ️ 未找到相關文件。"
    return retrieved_docs, "✅ 文件檢索完成。"

In [ ]:
def generate_rag_response_from_faiss(query_text, max_output_tokens=1024, temperature=0.2):
    retrieved_docs, msg = retrieve_documents_from_faiss(query_text)

    if not retrieved_docs:
        return msg, ""

    context_parts = []
    for doc in retrieved_docs:
        context_parts.append(
            f"### 來源文章標題: {doc['title']}\n"
            f"文章連結: {doc['url']}\n"
            f"文章內容: {doc['content']}\n"
        )
    context = "\n---\n".join(context_parts)

    prompt = (
        "你是一個善於從提供的參考文件中找出答案的智能助理。" \
        "請根據以下提供的參考文件內容來回答問題。" \
        "如果參考文件沒有足夠的資訊來回答問題，請直接回答 '我不知道'，不要編造資訊。" \
        "請確保你的回答是基於所提供的參考文件。\n\n"
        f"--- 參考文件開始 ---\n{context}\n--- 參考文件結束 ---\n\n"
        f"問題: {query_text}\n"
        "答案:"
    )

    try:
        response = model.generate_content(prompt, generation_config={
            "max_output_tokens": max_output_tokens,
            "temperature": temperature,
            "top_p": 0.8,
            "top_k": 40
        })
        return "✅ 生成回應成功。", response.text
    except Exception as e:
        return f"⚠️ Gemini 生成回應失敗: {e}", ""

In [ ]:
if not ptt_posts_df.empty:
    print(f"目前共有 {len(ptt_posts_df)} 篇文章")
    display(ptt_posts_df.head())
else:
    print("目前還沒有抓取到任何資料，請先在 Gradio介面的 Crawler 分頁執行爬蟲")

In [ ]:
import time

def run_ptt_rag_pipeline(pages=1, min_push=10, test_query="這份資料中提到的經典電影有哪些？"):
    try:
        print("=== Step 1: 執行 PTT 爬蟲 ===")
        status, df = crawl_ptt_movie(index_pages=pages, min_push=min_push)
        print(status)

        print("\n=== Step 2: 建立 FAISS 向量資料庫 ===")
        msg, index = build_faiss_index_from_ptt_data()
        print(msg)

        if index is None:
             print("❌ FAISS 建立失敗，無法繼續測試 RAG。")
             return

        print(f"\n=== Step 3: 測試 RAG 問答 ===")
        print(f"問題: {test_query}")
        status_rag, response = generate_rag_response_from_faiss(test_query)
        print(f"結果: {status_rag}")
        print(f"回答: {response}")

        print("\n=== Step 4: 測試範圍外提問 (預期回答：我不知道) ===")
        out_query = "現在火星上的天氣如何？"
        _, out_response = generate_rag_response_from_faiss(out_query)
        print(f"問題: {out_query}")
        print(f"回答: {out_response}")
    except Exception as e:
        print(f"❌ Pipeline 執行總體失敗: {e}")

# 啟動 Pipeline
run_ptt_rag_pipeline(pages=1, min_push=5)

In [ ]:
import gradio as gr
import pandas as pd
ws_ptt = ws_ptt_posts

# ---------------------------------------
# 主 Gradio 介面
# ---------------------------------------
with gr.Blocks(title="待辦清單＋番茄鐘＋PTT 電影 RAG 系統") as demo:
    gr.Markdown("# ✅ 待辦任務管理 與 PTT 電影版 RAG 系統")

    # Summary 顯示
    with gr.Row():
        btn_refresh = gr.Button("🔄 重新整理（Sheet → App）")
        out_summary = gr.Markdown(today_summary())

    # Tasks 分頁
    with gr.Tab("Tasks"):
        with gr.Row():
            with gr.Column(scale=2):
                task = gr.Textbox(label="任務名稱", placeholder="寫 HW3 報告 / 修正 SQL / …")
                priority = gr.Dropdown(["H","M","L"], value="M", label="優先級")
                est_min = gr.Number(value=25, label="預估時間（分鐘）", precision=0)
                due_date = gr.Textbox(label="到期日（YYYY-MM-DD，可空白）")
                labels = gr.Textbox(label="標籤（逗號分隔，可空白）")
                notes = gr.Textbox(label="備註（可空白）")
                planned_for = gr.Dropdown(["","today","tomorrow"], value="", label="規劃歸屬")
                btn_add = gr.Button("➕ 新增任務")
                msg_add = gr.Markdown()
            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(value=pd.DataFrame(), label="任務清單", interactive=False)
        with gr.Row():
            task_choice = gr.Dropdown(choices=list_task_choices(), label="選取任務（用於更新）")
            new_status = gr.Dropdown(["todo","in-progress","done"], value="in-progress", label="更新狀態")
            btn_update = gr.Button("✏️ 更新狀態")
            btn_done = gr.Button("✅ 直接標記完成")
            msg_update = gr.Markdown()

    # Pomodoro 分頁
    with gr.Tab("Pomodoro"):
        with gr.Row():
            sel_task = gr.Dropdown(choices=list_task_choices(), label="選擇任務")
            cycles = gr.Number(value=1, precision=0, label="番茄數")
        with gr.Row():
            btn_start_work = gr.Button("▶️ 開始工作")
            note_work = gr.Textbox(label="工作備註（可空白）")
            btn_end_work = gr.Button("⏹️ 結束工作並記錄")
        with gr.Row():
            btn_start_break = gr.Button("🍵 開始休息")
            note_break = gr.Textbox(label="休息備註（可空白）")
            btn_end_break = gr.Button("⏹️ 結束休息並記錄")
        msg_pomo = gr.Markdown()
        grid_logs = gr.Dataframe(value=pd.DataFrame(), label="番茄鐘紀錄", interactive=False)

    # AI Plan 分頁
    with gr.Tab("AI Plan"):
        gr.Markdown("把**今天的任務**排成 **morning / afternoon / evening** 三段行動計畫。")
        btn_plan = gr.Button("🧠 產生今日計畫")
        out_plan = gr.Markdown()

    # Crawler 分頁
    with gr.Tab("Crawler"):
        url = gr.Textbox(label="目標 URL", placeholder="https://example.com")
        selector = gr.Textbox(label="CSS Selector", placeholder="a.news-item")
        mode = gr.Radio(["text","href","both"], value="text", label="擷取內容")
        limit = gr.Number(value=20, precision=0, label="最多擷取幾筆")
        btn_crawl = gr.Button("🕷️ 開始擷取")
        msg_crawl = gr.Markdown()
        grid_clips = gr.Dataframe(value=pd.DataFrame(), label="擷取結果", interactive=True)

    # PTT 爬蟲分頁
    with gr.Tab("PTT 爬蟲"):
        gr.Markdown("### 🕷️ PTT 電影版文章爬蟲")
        pages_input = gr.Number(value=3, label="往前爬取頁數", precision=0)
        btn_build_rag = gr.Button("🚀 開始爬取 PTT", variant="primary")
        out_rag_status = gr.Markdown("ℹ️ 系統狀態")

    # RAG 分頁
    with gr.Tab("RAG (PTT Data)"):
        gr.Markdown("### 💬 RAG 知識庫問答")
        btn_faiss = gr.Button("🛠️ 建立/更新 FAISS 索引")
        rag_query = gr.Textbox(label="輸入問題", placeholder="關於電影版最近的討論…", lines=2)
        btn_rag_chat = gr.Button("💡 RAG 提問")
        out_rag_answer = gr.Markdown("### 【AI 助理的回答將會顯示於此】")

    # Summary 分頁
    with gr.Tab("Summary"):
        btn_summary = gr.Button("📊 重新計算今日完成率")
        out_summary2 = gr.Markdown()

    # ---------------------------------------
    # 事件綁定
    # ---------------------------------------
    btn_refresh.click(lambda: (pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), list_task_choices(), today_summary()), outputs=[grid_tasks, grid_logs, grid_clips, task_choice, out_summary])
    btn_add.click(add_task, inputs=[task, priority, est_min, due_date, labels, notes, planned_for], outputs=[msg_add, grid_tasks])
    btn_update.click(update_task_status, inputs=[task_choice, new_status], outputs=[msg_update, grid_tasks])
    btn_done.click(mark_done, inputs=[task_choice], outputs=[msg_update, grid_tasks])
    btn_start_work.click(start_phase, inputs=[sel_task, gr.State("work"), cycles], outputs=[msg_pomo])
    btn_end_work.click(end_phase, inputs=[sel_task, note_work], outputs=[msg_pomo])
    btn_start_break.click(start_phase, inputs=[sel_task, gr.State("break"), cycles], outputs=[msg_pomo])
    btn_end_break.click(end_phase, inputs=[sel_task, note_break], outputs=[msg_pomo])
    btn_plan.click(generate_today_plan, outputs=[out_plan])
    btn_summary.click(today_summary, outputs=[out_summary2])

    # PTT 爬蟲事件綁定（修改後）
    def _crawl_ptt(pages, keyword=None):
        yield "⏳ 正在爬取 PTT 電影版文章..."
        try:
            status_msg_crawl, new_df = crawl_ptt_movie(index_pages=pages, delay=1.5, keyword=keyword)
            if new_df.empty or "沒有新文章" in status_msg_crawl:
                yield "⚠️ 本次未抓到文章或沒有新文章符合條件，將使用舊資料。"
                old_df = read_df(ws_ptt, PTT_HEADER)
                status_msg_faiss, _ = build_faiss_index_from_ptt_data()
                yield f"ℹ️ 已載入舊資料建立索引：\n{status_msg_faiss}"
                return

            old_df = read_df(ws_ptt, PTT_HEADER)
            combined_df = pd.concat([old_df, new_df]).drop_duplicates(subset=["post_id"]).sort_values(by="fetched_at", ascending=False)
            write_df(ws_ptt, combined_df, PTT_HEADER)
            status_msg_faiss, _ = build_faiss_index_from_ptt_data() # Rebuild FAISS with combined data
            yield f"✅ 成功更新索引！\n{status_msg_faiss}"
        except Exception as e:
            yield f"❌ 發生錯誤：{e}"

    btn_build_rag.click(_crawl_ptt, inputs=[pages_input], outputs=[out_rag_status])

    # RAG 問答事件綁定
    def _rag_query_func(q):
        try:
            # Corrected function call to generate_rag_response_from_faiss
            status, answer = generate_rag_response_from_faiss(q, max_output_tokens=1024, temperature=0.2)
            if "無法取得回答" in status or "不知道" in answer:
                return "⚠️ 無法取得回答，請先確認索引已建立。" + answer
            return answer
        except Exception as e:
            return f"⚠️ 查詢失敗：{e}，請先確認索引已建立。"

    btn_rag_chat.click(_rag_query_func, inputs=[rag_query], outputs=[out_rag_answer])

# 啟動 Gradio
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b69023463661b564b4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#=========================
# Gradio 介面
# =========================
def _refresh():
    global tasks_df, logs_df, clips_df, ptt_posts_df
    tasks_df, logs_df, clips_df = refresh_all()
    ptt_posts_df = read_ptt_posts_df()
    return tasks_df, logs_df, clips_df, list_task_choices(), today_summary(), ptt_posts_df

with gr.Blocks(title="待辦清單＋番茄鐘＋AI 計畫（Sheet/Gradio/爬蟲）") as demo:
    gr.Markdown("# ✅ 待辦清單與番茄鐘（Google Sheet＋Gradio＋Crawler＋AI 計畫）")
    with gr.Row():
        btn_refresh = gr.Button("🔄 重新整理（Sheet → App）")
        out_summary = gr.Markdown(today_summary())

    with gr.Tab("Tasks"):
        with gr.Row():
            with gr.Column(scale=2):
                task = gr.Textbox(label="任務名稱", placeholder="寫 HW3 報告 / 修正 SQL / …")
                priority = gr.Dropdown(["H","M","L"], value="M", label="優先級")
                est_min = gr.Number(value=25, label="預估時間（分鐘）", precision=0)
                due_date = gr.Textbox(label="到期日（YYYY-MM-DD，可空白）")
                labels = gr.Textbox(label="標籤（逗號分隔，可空白）")
                notes = gr.Textbox(label="備註（可空白）")
                planned_for = gr.Dropdown(["","today","tomorrow"], value="", label="規劃歸屬")
                btn_add = gr.Button("➕ 新增任務")
                msg_add = gr.Markdown()
            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(value=tasks_df, label="任務清單（直接從 Sheet 來）", interactive=False)

        with gr.Row():
            task_choice = gr.Dropdown(choices=list_task_choices(), label="選取任務（用於更新）")
            new_status = gr.Dropdown(["todo","in-progress","done"], value="in-progress", label="更新狀態")
            btn_update = gr.Button("✏️ 更新狀態")
            btn_done = gr.Button("✅ 直接標記完成")
            msg_update = gr.Markdown()

    with gr.Tab("Pomodoro"):
        with gr.Row():
            sel_task = gr.Dropdown(choices=list_task_choices(), label="選擇任務")
            cycles = gr.Number(value=1, precision=0, label="番茄數（僅作紀錄）")
        with gr.Row():
            btn_start_work = gr.Button("▶️ 開始工作")
            note_work = gr.Textbox(label="工作備註（可空白）")
            btn_end_work = gr.Button("⏹️ 結束工作並記錄")
        with gr.Row():
            btn_start_break = gr.Button("🍵 開始休息")
            note_break = gr.Textbox(label="休息備註（可空白）")
            btn_end_break = gr.Button("⏹️ 結束休息並記錄")
        msg_pomo = gr.Markdown()
        grid_logs = gr.Dataframe(value=logs_df, label="番茄鐘紀錄", interactive=False)

    with gr.Tab("AI Plan"):
        gr.Markdown("把**今天的任務**排成 **morning / afternoon / evening** 三段行動計畫。")
        btn_plan = gr.Button("🧠 產生今日計畫")
        out_plan = gr.Markdown()

    with gr.Tab("Crawler"):
        url = gr.Textbox(label="目標 URL", placeholder="https://example.com")
        selector = gr.Textbox(label="CSS Selector", placeholder="a.news-item / h2.title")
        mode = gr.Radio(["text","href","both"], value="text", label="擷取內容")
        limit = gr.Number(value=20, precision=0, label="最多擷取幾筆")
        btn_crawl = gr.Button("🕷️ 開始擷取")
        msg_crawl = gr.Markdown()
        grid_clips = gr.Dataframe(value=clips_df, label="擷取結果", interactive=True)
        btn_add_clips = gr.Button("➕ 將勾選的項目加入為任務")

    with gr.Tab("PTT 爬蟲"):
        gr.Markdown("### PTT 電影版文章爬蟲")
        ptt_pages = gr.Number(value=3, label="往前爬取頁數", precision=0)
        ptt_min_push = gr.Number(value=0, label="最少推文數", precision=0)
        ptt_kw = gr.Textbox(label="關鍵字篩選")
        btn_crawl_ptt = gr.Button("🚀 開始爬取 PTT")
        msg_ptt_crawl = gr.Markdown()
        grid_ptt = gr.Dataframe(value=ptt_posts_df, label="PTT 資料內容")

    with gr.Tab("RAG (PTT Data)"):
        gr.Markdown("### RAG 知識庫問答")
        btn_build_index = gr.Button("🛠️ 建立/更新 FAISS 索引")
        msg_index = gr.Markdown()
        rag_input = gr.Textbox(label="輸入問題", placeholder="關於電影版最近的討論...")
        btn_rag = gr.Button("💡 RAG 提問")
        rag_output = gr.Markdown()

    with gr.Tab("Summary"):
        btn_summary = gr.Button("📊 重新計算今日完成率")
        out_summary2 = gr.Markdown()

    # === 綁定動作 ===
    btn_refresh.click(_refresh, outputs=[grid_tasks, grid_logs, grid_clips, task_choice, out_summary, grid_ptt])
    btn_add.click(add_task, inputs=[task, priority, est_min, due_date, labels, notes, planned_for], outputs=[msg_add, grid_tasks])
    btn_update.click(update_task_status, inputs=[task_choice, new_status], outputs=[msg_update, grid_tasks])
    btn_done.click(mark_done, inputs=[task_choice], outputs=[msg_update, grid_tasks])
    btn_start_work.click(start_phase, inputs=[sel_task, gr.State("work"), cycles], outputs=[msg_pomo])
    btn_end_work.click(end_phase, inputs=[sel_task, note_work], outputs=[msg_pomo])
    btn_plan.click(generate_today_plan, outputs=[out_plan])

    btn_crawl.click(crawl, inputs=[url, selector, mode, limit], outputs=[grid_clips, msg_crawl])
    btn_crawl_ptt.click(crawl_ptt_movie, inputs=[ptt_pages, ptt_min_push, ptt_kw], outputs=[msg_ptt_crawl, grid_ptt])

    btn_build_index.click(build_faiss_index_from_ptt_data, outputs=[msg_index])
    btn_rag.click(generate_rag_response_from_faiss, inputs=[rag_input], outputs=[msg_index, rag_output])

    btn_summary.click(today_summary, outputs=[out_summary2])

# 強制開啟 share=True 以在 Colab 獲取公開網址
demo.launch(share=True, inline=False)